# Demo: (Optional) Basic ETL with the DataFrame API

Databricks free https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/3801390934062894?o=7474657872577658

aula trilha: https://community.databricks.com/t5/learning-events/advanced-learning-festival-15-june-06-july-2026/ev-p/156932

https://community.databricks.com/t5/learning-events/virtual-learning-festival-10-october-31-october-2025/ev-p/127652

https://customer-academy.databricks.com/learn/courses/3962/developing-applications-with-apache-spark?hash=d2245ade317149fcb8f6d57ad991d9aa7f055f9f&generated_by=409740

https://spark.apache.org/docs/latest/api/sql/index.html

# databricks Academy

## 1.2 Optional Demo - Basic ETL with the DataFrame API

This optional demonstration walks through common ETL operations using the Bakehouse sales transactions sample dataset. We cover data loading, validation, transformation, and enrichment using the DataFrame API. If you are comfortable with this material from the Introduction to Apache Spark course, you can skim it as a refresher.

### Learning Objectives

By the end of this demo, you will be able to:

* Implement common ETL operations using Spark DataFrames
* Validate data quality with both Spark SQL and the DataFrame API
* Create derived features through transformations
* Use different column reference methods
* Work with user defined functions (UDFs)

## REQUIRED - SELECT CLASSIC COMPUTE

### Select Compute

Before starting this notebook, select your Classic Compute cluster from the compute drop-down at the top-right of the notebook. Be aware that Serverless is selected by default.

* If your cluster is not shown in the drop-down, select **More**, then choose your cluster in the **Attach to an existing compute resource** dialog.
* If your cluster has terminated, open **Compute** in the left navigation, start your cluster, then return here and attach it.

> **NOTE:** This course was developed and tested on Databricks Runtime 17.3 LTS (Apache Spark 4.0). Several demonstrations reference the Spark UI and query execution plans, which are not exposed on Serverless compute.

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **0 - Required Setup** notebook has been run, sets your default catalog, and sets the default schema to `spark_dev`.



In [0]:
#python
#%run ../Includes/Classroom-Setup-1

## Transaction Data Processing Requirements

We are going to act on a typical ETL requirement: land a validated, enriched version of the raw transaction data as a new table.

* **Source data** - the `samples.bakehouse.sales_transactions` table (point-of-sale transactions from the Bakehouse sample dataset)
* **Target** - a table named `transactions_enriched` with this schema:

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| transactionID | BIGINT | Transaction identifier |
| datetime | TIMESTAMP | When the transaction occurred |
| product | STRING | Product sold |
| quantity | BIGINT | Units sold |
| totalPrice | BIGINT | Total sale amount |
| paymentMethod | STRING | Payment method, normalized to uppercase |
| cardNumberMasked | STRING | Card number with all but the last 4 digits masked |
| orderSize | STRING | One of `bulk`, `large`, `medium`, `small`, based on quantity |

---

## B. Data Loading and Inspection

First, let's load and inspect the data.

In [0]:
# Read the Transactions Table
transactions_df = spark.read.table("samples.bakehouse.sales_transactions")

In [0]:
# Print the Schema
transactions_df.printSchema()

root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



### What to Observe

The data includes fields such as transactionID, dateTime, product, quantity, totalPrice, and cardNumber.

Notice that cardNumber contains full numeric values, which will need to be masked for security and data quality during the enrichment steps.

What to Observe

The schema was read from the table's Unity Catalog metadata - no inference required. Note that cardNumber arrived as a BIGINT : storing a card number as a number is a data quality smell, and we will convert and mask it during enrichment.

In [0]:
# Display the First Few Rows
display(transactions_df.limit(5))

transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
1002961,2000253,3000047,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,3,24,amex,378154478982993
1003007,2000226,3000047,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
1003017,2000108,3000047,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
1003068,2000173,3000047,2024-05-02T04:31:51.612Z,Pearly Pies,28,3,84,amex,343808569426192
1003103,2000075,3000047,2024-05-04T23:44:26.902Z,Pearly Pies,28,3,84,visa,4377080942201798


In [0]:
# Visually Inspect a Subset of the Data
display(transactions_df.limit(10))

transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
1002961,2000253,3000047,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,3,24,amex,378154478982993
1003007,2000226,3000047,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,3,108,mastercard,2244626981238094
1003017,2000108,3000047,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,3,120,mastercard,2490570234487424
1003068,2000173,3000047,2024-05-02T04:31:51.612Z,Pearly Pies,28,3,84,amex,343808569426192
1003103,2000075,3000047,2024-05-04T23:44:26.902Z,Pearly Pies,28,3,84,visa,4377080942201798
1003147,2000295,3000047,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,3,96,amex,371093774812677
1003196,2000237,3000047,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,3,120,mastercard,5538807345848392
1003329,2000272,3000047,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,3,84,visa,4872480716880043
1001264,2000209,3000047,2024-05-16T17:32:28.547Z,Pearly Pies,28,3,84,mastercard,5287105980593305
1001287,2000120,3000047,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,3,120,amex,376211012259783


In [0]:
# Select Only the Columns We Need
# Remember: filter early, filter often
transactions_required_cols_df = transactions_df.select(
    "transactionID",
    "dateTime",
    "product",
    "quantity",
    "unitPrice",
    "totalPrice",
    "paymentMethod",
    "cardNumber"
)

In [0]:
# Alternatively we could have used the drop() method to remove the columns we didn't want

In [0]:
# Count the Source Records
initial_count = transactions_required_cols_df.count()

print(f"Source data has {initial_count} records")

Source data has 3333 records


## C. Data Validation

Before enriching the data we check it for nulls and internal consistency—here, that the line total equals quantity * unitPrice.   

ANSI Mode and Casting   


Spark 4 runs in ANSI SQL mode by default: CAST('NA' AS INT) raises an error instead of silently returning NULL. The Bakehouse sample data is clean, but when you process data of unknown quality, use TRY_CAST, which returns NULL for invalid values, letting you find and handle them deliberately.   

In [0]:
# Register a Temp View for SQL Analysis
transactions_required_cols_df.createOrReplaceTempView("transactions_temp")

In [0]:
transactions_required_cols_df.printSchema()

root
 |-- transactionID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



In [0]:
# Count Null Values with Spark SQL
# Use the COUNT_IF SQL function to count null values per column
null_counts_sql = spark.sql("""
    SELECT
        COUNT_IF(transactionID IS NULL) AS Null_TransactionID_Count,
        COUNT_IF(dateTime IS NULL) AS Null_DateTime_Count,
        COUNT_IF(product IS NULL) AS Null_Product_Count,
        COUNT_IF(quantity IS NULL) AS Null_Quantity_Count,
        COUNT_IF(totalPrice IS NULL) AS Null_TotalPrice_Count,
        COUNT_IF(paymentMethod IS NULL) AS Null_PaymentMethod_Count,
        COUNT_IF(cardNumber IS NULL) AS Null_CardNumber_Count
    FROM transactions_temp
""")

display(null_counts_sql)

Null_TransactionID_Count,Null_DateTime_Count,Null_Product_Count,Null_Quantity_Count,Null_TotalPrice_Count,Null_PaymentMethod_Count,Null_CardNumber_Count
0,0,0,0,0,0,0


In [0]:
# Count Null Values with Spark SQL
# Use the COUNT_IF SQL function to count null values per column
null_counts_sql = spark.sql("""
    SELECT
        COUNT_IF(transactionID IS NULL) AS Null_TransactionID_Count,
        COUNT_IF(dateTime IS NULL) AS Null_DateTime_Count,
        COUNT_IF(product IS NULL) AS Null_Product_Count,
        COUNT_IF(quantity IS NULL) AS Null_Quantity_Count,
        COUNT_IF(unitPrice IS NULL) AS Null_UnitPrice_Count,
        COUNT_IF(totalPrice IS NULL) AS Null_TotalPrice_Count,
        COUNT_IF(paymentMethod IS NULL) AS Null_PaymentMethod_Count,
        COUNT_IF(cardNumber IS NULL) AS Null_CardNumber_Count
    FROM transactions_temp
""")

display(null_counts_sql)

Null_TransactionID_Count,Null_DateTime_Count,Null_Product_Count,Null_Quantity_Count,Null_UnitPrice_Count,Null_TotalPrice_Count,Null_PaymentMethod_Count,Null_CardNumber_Count
0,0,0,0,0,0,0,0


## Comparing Spark SQL to DataFrame API Operations

Spark SQL queries and their equivalent operations in the DataFrame API are evaluated to the same physical plans. Let's prove this.

In [0]:
# Count Null Values with the DataFrame API
# This is the equivalent of the preceding Spark SQL query using the DataFrame API
from pyspark.sql.functions import col, sum, when

# Make sure to work with the same temporary view that the SQL is using
transactions_temp_df = spark.table("transactions_temp")

null_counts_df = transactions_temp_df.select(
    sum(when(col("transactionID").isNull(), 1).otherwise(0)).alias("Null_TransactionID_Count"),
    sum(when(col("dateTime").isNull(), 1).otherwise(0)).alias("Null_DateTime_Count"),
    sum(when(col("product").isNull(), 1).otherwise(0)).alias("Null_Product_Count"),
    sum(when(col("quantity").isNull(), 1).otherwise(0)).alias("Null_Quantity_Count"),
    sum(when(col("unitPrice").isNull(), 1).otherwise(0)).alias("Null_UnitPrice_Count"),
    sum(when(col("totalPrice").isNull(), 1).otherwise(0)).alias("Null_TotalPrice_Count"),
    sum(when(col("paymentMethod").isNull(), 1).otherwise(0)).alias("Null_PaymentMethod_Count"),
    sum(when(col("cardNumber").isNull(), 1).otherwise(0)).alias("Null_CardNumber_Count")
)

display(null_counts_df)

Null_TransactionID_Count,Null_DateTime_Count,Null_Product_Count,Null_Quantity_Count,Null_UnitPrice_Count,Null_TotalPrice_Count,Null_PaymentMethod_Count,Null_CardNumber_Count
0,0,0,0,0,0,0,0


In [0]:
# Explain the SQL Version
null_counts_sql.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonAgg(keys=[], functions=[count(if (isnotnull(transactionID#11121L)) null else isnull(transactionID#11121L)), count(if (isnotnull(dateTime#11124)) null else isnull(dateTime#11124)), count(if (isnotnull(product#11125)) null else isnull(product#11125)), count(if (isnotnull(quantity#11126L)) null else isnull(quantity#11126L)), count(if (isnotnull(unitPrice#11127L)) null else isnull(unitPrice#11127L)), count(if (isnotnull(totalPrice#11128L)) null else isnull(totalPrice#11128L)), count(if (isnotnull(paymentMethod#11129)) null else isnull(paymentMethod#11129)), count(if (isnotnull(cardNumber#11130L)) null else isnull(cardNumber#11130L))])
         +- PhotonScan parquet samples.bakehouse.sales_transactions[transactionID#11121L,dateTime#11124,product#11125,quantity#11126L,unitPrice#11127L,totalPrice#11128L,paymentMethod#11129,cardNumber#11130L] DataFilters: 

### What to Observe

Compare the two physical plans printed above — they are the same. Spark SQL and the DataFrame API are two front ends to the same Catalyst optimizer and execution engine, so you can choose whichever is clearer for the task at hand.

In [0]:
# Check Line Total Consistency
from pyspark.sql.functions import col

# Keep only rows where the line total is consistent with quantity * unitPrice
valid_transactions_df = transactions_required_cols_df.filter(
    col("quantity") > 0
).filter(
    col("totalPrice") == col("quantity") * col("unitPrice")
)

print(f"{valid_transactions_df.count()} of {initial_count} records passed validation")

3333 of 3333 records passed validation


In [0]:
valid_transactions_df.show(5)

+-------------+--------------------+--------------------+--------+---------+----------+-------------+----------------+
|transactionID|            dateTime|             product|quantity|unitPrice|totalPrice|paymentMethod|      cardNumber|
+-------------+--------------------+--------------------+--------+---------+----------+-------------+----------------+
|      1002961|2024-05-14 12:17:...|  Golden Gate Ginger|       8|        3|        24|         amex| 378154478982993|
|      1003007|2024-05-10 23:10:...|Austin Almond Bis...|      36|        3|       108|   mastercard|2244626981238094|
|      1003017|2024-05-16 16:34:...|Austin Almond Bis...|      40|        3|       120|   mastercard|2490570234487424|
|      1003068|2024-05-02 04:31:...|         Pearly Pies|      28|        3|        84|         amex| 343808569426192|
|      1003103|2024-05-04 23:44:...|         Pearly Pies|      28|        3|        84|         visa|4377080942201798|
+-------------+--------------------+------------

## D. Data Enrichment

Now let's create the derived columns from the requirements: a normalized payment method, a masked card number, and an order size category.

In [0]:
# Normalize the Payment Method
from pyspark.sql.functions import upper, col

normalized_df = valid_transactions_df.withColumn(
    "paymentMethod", upper(col("paymentMethod"))
)

display(normalized_df.select("transactionID", "paymentMethod").limit(10))

transactionID,paymentMethod
1002961,AMEX
1003007,MASTERCARD
1003017,MASTERCARD
1003068,AMEX
1003103,VISA
1003147,AMEX
1003196,MASTERCARD
1003329,VISA
1001264,MASTERCARD
1001287,AMEX


In [0]:
# Mask the Card Number
from pyspark.sql.functions import col, concat, lit, right

# Cast the card number to a string, then keep only the last 4 digits
masked_df = normalized_df.withColumn(
    "cardNumberMasked",
    concat(lit("****-"), right(col("cardNumber").cast("string"), lit(4)))
).drop("cardNumber")

display(masked_df.select("transactionID", "cardNumberMasked").limit(10))

transactionID,cardNumberMasked
1002961,****-2993
1003007,****-8094
1003017,****-7424
1003068,****-6192
1003103,****-1798
1003147,****-2677
1003196,****-8392
1003329,****-0043
1001264,****-3305
1001287,****-9783


In [0]:
# Mask the Card Number
from pyspark.sql.functions import col, concat, lit, left, right

# Cast the card number to a string, then keep only the last 4 digits
masked2_df = normalized_df.withColumn(
    "cardNumberMasked",
    concat(left(col("cardNumber").cast("string"), lit(5)), lit("-****"))
)

display(masked2_df.select("transactionID", "cardNumberMasked","cardNumber").limit(10))

transactionID,cardNumberMasked,cardNumber
1002961,37815-****,378154478982993
1003007,22446-****,2244626981238094
1003017,24905-****,2490570234487424
1003068,34380-****,343808569426192
1003103,43770-****,4377080942201798
1003147,37109-****,371093774812677
1003196,55388-****,5538807345848392
1003329,48724-****,4872480716880043
1001264,52871-****,5287105980593305
1001287,37621-****,376211012259783


In [0]:
# Categorize the Order Size
from pyspark.sql.functions import when, col

# Categorize the quantity column into: Bulk, Large, Medium, Small
enriched_df = masked_df.withColumn(
    "orderSize",
    when(col("quantity") >= 24, "Bulk")
    .when(col("quantity") >= 12, "Large")
    .when(col("quantity") >= 6, "Medium")
    .otherwise("Small")
).drop("unitPrice")

display(enriched_df.limit(10))

transactionID,dateTime,product,quantity,totalPrice,paymentMethod,cardNumberMasked,orderSize
1002961,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,24,AMEX,****-2993,Medium
1003007,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,108,MASTERCARD,****-8094,Bulk
1003017,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,120,MASTERCARD,****-7424,Bulk
1003068,2024-05-02T04:31:51.612Z,Pearly Pies,28,84,AMEX,****-6192,Bulk
1003103,2024-05-04T23:44:26.902Z,Pearly Pies,28,84,VISA,****-1798,Bulk
1003147,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,96,AMEX,****-2677,Bulk
1003196,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,120,MASTERCARD,****-8392,Bulk
1003329,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,84,VISA,****-0043,Bulk
1001264,2024-05-16T17:32:28.547Z,Pearly Pies,28,84,MASTERCARD,****-3305,Bulk
1001287,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,120,AMEX,****-9783,Bulk


## Visualize the Order Sizes

PySpark in Spark 4 includes native plotting: call `df.plot` directly on a DataFrame (backed by plotly) without converting to pandas first.

In [0]:
%pip install plotly


Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 117.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

# Plot Order Sizes with Native Plotting
# Aggregate, then plot directly from the PySpark DataFrame (new in Spark 4)
order_size_counts_df = enriched_df.groupBy("orderSize").count()

order_size_counts_df.plot.bar(x="orderSize", y="count")

<!doctype html>

In [0]:
# Plot Order Sizes and Save as HTML
import os

order_size_counts_df = enriched_df.groupBy("orderSize").count()

# Native PySpark plotting returns a plotly Figure (new in Spark 4)
fig = order_size_counts_df.plot.bar(x="orderSize", y="count")

# Save the figure as a standalone, interactive HTML file
html_path = os.path.join(os.getcwd(), "order_size_counts_aula1_ETLSPARK.html")
fig.write_html(html_path)

print(f"Plot saved to {html_path}")
fig.show()

Plot saved to /Workspace/Users/fabieneaulas@gmail.com/APACHE_SPARK/DevelopingApplicationswithApacheSpark/order_size_counts_aula1_ETLSPARK.html


<!doctype html>

> **Other Visualization Options**
> 
> You can also use the **+ > Visualization** option in any `display()` result, or ask the Databricks Assistant to generate chart code for you.

---

## E. Working with Different Column References

Let's analyze the enriched data while exercising the column referencing approaches from the lecture.

In [0]:
# Reference Columns Directly
# Direct string references
display(enriched_df.select("product", "orderSize").limit(100))

product,orderSize
Golden Gate Ginger,Medium
Austin Almond Biscotti,Bulk
Austin Almond Biscotti,Bulk
Pearly Pies,Bulk
Pearly Pies,Bulk
Austin Almond Biscotti,Bulk
Tokyo Tidbits,Bulk
Outback Oatmeal,Bulk
Pearly Pies,Bulk
Austin Almond Biscotti,Bulk


In [0]:
# Reference Columns with Column Objects
from pyspark.sql.functions import col

# Column objects support methods like alias()
display(
    enriched_df
    .select(col("product").alias("product_name"), col("totalPrice"))
    .limit(100)
)

product_name,totalPrice
Golden Gate Ginger,24
Austin Almond Biscotti,108
Austin Almond Biscotti,120
Pearly Pies,84
Pearly Pies,84
Austin Almond Biscotti,96
Tokyo Tidbits,120
Outback Oatmeal,84
Pearly Pies,84
Austin Almond Biscotti,120


In [0]:
# Reference Columns with String Expressions
# selectExpr accepts SQL expressions
display(
    enriched_df
    .selectExpr("product", "totalPrice", "totalPrice > 100 AS isLargeSale")
    .limit(100)
)

product,totalPrice,isLargeSale
Golden Gate Ginger,24,false
Austin Almond Biscotti,108,true
Austin Almond Biscotti,120,true
Pearly Pies,84,false
Pearly Pies,84,false
Austin Almond Biscotti,96,false
Tokyo Tidbits,120,true
Outback Oatmeal,84,false
Pearly Pies,84,false
Austin Almond Biscotti,120,true


## F. Working with UDFs

Let's use a vectorized pandas UDF to calculate the z-score (standard deviations from the mean) of the total price for each transaction.

In [0]:
# Apply a Vectorized Pandas UDF
from pyspark.sql.functions import pandas_udf

# Pandas UDF (vectorized) - operates on batches of rows as pandas Series
@pandas_udf("double")
def normalized_price(price_series):
    return (price_series - price_series.mean()) / price_series.std()

udf_example_df = enriched_df.withColumn(
    "price_normalized", normalized_price("totalPrice")
)

display(udf_example_df.limit(100))

# Note: in practice, prefer built-in functions over UDFs when possible

transactionID,dateTime,product,quantity,totalPrice,paymentMethod,cardNumberMasked,orderSize,price_normalized
1002961,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,24,AMEX,****-2993,Medium,-1.4599544129211217
1003007,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,108,MASTERCARD,****-8094,Bulk,0.3748177397625141
1003017,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,120,MASTERCARD,****-7424,Bulk,0.6369280472887479
1003068,2024-05-02T04:31:51.612Z,Pearly Pies,28,84,AMEX,****-6192,Bulk,-0.14940287528995327
1003103,2024-05-04T23:44:26.902Z,Pearly Pies,28,84,VISA,****-1798,Bulk,-0.14940287528995327
1003147,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,96,AMEX,****-2677,Bulk,0.11270743223628042
1003196,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,120,MASTERCARD,****-8392,Bulk,0.6369280472887479
1003329,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,84,VISA,****-0043,Bulk,-0.14940287528995327
1001264,2024-05-16T17:32:28.547Z,Pearly Pies,28,84,MASTERCARD,****-3305,Bulk,-0.14940287528995327
1001287,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,120,AMEX,****-9783,Bulk,0.6369280472887479


## G. Putting It All Together

Let's combine everything into a single chained operation that reads from the source, validates, enriches, and saves to a new target table (overwriting any existing data).

In [0]:
%sql
-- Drop the Target Table if it Exists
DROP TABLE IF EXISTS transactions_enriched;

In [0]:
# Run the Full ETL Pipeline
from pyspark.sql.functions import col, concat, lit, right, upper, when

(
    spark.read.table("samples.bakehouse.sales_transactions")
    # Project the columns we need
    .select(
        "transactionID",
        "dateTime",
        "product",
        "quantity",
        "unitPrice",
        "totalPrice",
        "paymentMethod",
        "cardNumber",
    )
    # Keep only internally consistent rows
    .filter((col("quantity") > 0) & (col("totalPrice") == col("quantity") * col("unitPrice")))
    # Normalize the payment method
    .withColumn("paymentMethod", upper(col("paymentMethod")))
    # Mask the card number
    .withColumn(
        "cardNumberMasked",
        concat(lit("****-"), right(col("cardNumber").cast("string"), lit(4))),
    )
    .drop("cardNumber")
    # Categorize the order size
    .withColumn(
        "orderSize",
        when(col("quantity") >= 24, "Bulk")
        .when(col("quantity") >= 12, "Large")
        .when(col("quantity") >= 6, "Medium")
        .otherwise("Small"),
    )
    .drop("unitPrice")
    # Write to the target table
    .write
    .mode("overwrite")
    .saveAsTable("transactions_enriched")
)

In [0]:
%sql
-- Query the target table
SELECT * FROM transactions_enriched LIMIT 10;

transactionID,dateTime,product,quantity,totalPrice,paymentMethod,cardNumberMasked,orderSize
1002961,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,24,AMEX,****-2993,Medium
1003007,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,108,MASTERCARD,****-8094,Bulk
1003017,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,120,MASTERCARD,****-7424,Bulk
1003068,2024-05-02T04:31:51.612Z,Pearly Pies,28,84,AMEX,****-6192,Bulk
1003103,2024-05-04T23:44:26.902Z,Pearly Pies,28,84,VISA,****-1798,Bulk
1003147,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,96,AMEX,****-2677,Bulk
1003196,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,120,MASTERCARD,****-8392,Bulk
1003329,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,84,VISA,****-0043,Bulk
1001264,2024-05-16T17:32:28.547Z,Pearly Pies,28,84,MASTERCARD,****-3305,Bulk
1001287,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,120,AMEX,****-9783,Bulk


## Key Takeaways

### Data validation best practices:
* Project only the columns you need, as early as possible.
* Check for nulls and internal consistency before enriching.
* Under Spark 4 ANSI mode, use `try_cast()` when converting data of unknown quality.

### Data enrichment:
* Create meaningful derived columns driven by business requirements.
* Use built-in functions where possible, and user defined functions where necessary.

### Pipeline composition:
* Because transformations are lazy and return new DataFrames, an entire pipeline can be expressed as one readable chain ending in a single write action.